# 05 · Multimodal Acquisition, Calibration, and Manifests

**Where this sits in PyTex.** EBSD, XRD, and TEM data all share the same underlying
questions: *which frames are involved, how is the instrument calibrated, and how good
is the measurement?* PyTex answers these once, in a shared **acquisition layer**, and
serialises the answer into a **schema-versioned manifest**. This is the backbone of
the library's reproducibility and provenance doctrine: a workflow that crosses a tool
boundary must carry a machine-readable record of how the data were obtained.

## Learning goals

1. assemble an `AcquisitionGeometry` that ties frames, modality, calibration, and quality together;
2. attach a `CalibrationRecord` and `MeasurementQuality` to make instrument state explicit;
3. promote the acquisition to an `ExperimentManifest` and **serialise it to JSON**;
4. **round-trip** the manifest and validate it against its published schema.

## Why a manifest, not a naming convention

A file named `sample3_rerun_final.ang` records nothing a machine can trust. PyTex
instead makes the experiment self-describing: the manifest states the schema it obeys,
the PyTex version and canonical convention set that produced it, the physical frames,
the specimen→map/detector transforms, the calibration provenance, and the quality
metadata — in one validated document. Downstream code (and a future you) can read it
without guessing. (See the *data contracts and manifests* standard.)

In [ ]:
from __future__ import annotations

import json
import tempfile
import warnings
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    AcquisitionGeometry,
    CalibrationRecord,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    MeasurementQuality,
    ReferenceFrame,
    get_phase_fixture,
    read_experiment_manifest,
    validate_experiment_manifest,
)

# The frames an EBSD acquisition actually involves.
specimen = ReferenceFrame("specimen", FrameDomain.SPECIMEN, ("x", "y", "z"))
map_frame = ReferenceFrame("map", FrameDomain.MAP, ("i", "j", "k"))
crystal = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)

## 1 · Acquisition geometry: frames + modality + state

`AcquisitionGeometry` is the in-memory description of *how a dataset was collected*.
Here we describe an EBSD scan: the specimen frame, the map frame the pixels live in,
the rigid `specimen_to_map` transform (Notebook 01), and — crucially — the calibration
and quality records. `CalibrationRecord` states *where the calibration came from* and
its residual; `MeasurementQuality` carries confidence and valid-fraction so consumers
can filter bad pixels without re-deriving them.

In [ ]:
acquisition = AcquisitionGeometry(
    specimen_frame=specimen,
    modality="ebsd",
    map_frame=map_frame,
    specimen_to_map=FrameTransform(
        source=specimen, target=map_frame, rotation_matrix=np.eye(3),
    ),
    calibration_record=CalibrationRecord(
        source="stage-fit", status="calibrated", residual_error=0.1,
    ),
    measurement_quality=MeasurementQuality(confidence=0.92, valid_fraction=0.98),
)

print("modality           :", acquisition.modality)
print("map frame domain   :", acquisition.map_frame.domain.value)
print("calibration status :", acquisition.calibration_record.status,
      f"(residual {acquisition.calibration_record.residual_error})")
print("mean confidence    :", acquisition.measurement_quality.confidence)

## 2 · Promoting to a manifest and serialising

`ExperimentManifest.from_acquisition_geometry` freezes the acquisition into a
serialisable record, stamping it with a schema id/version, the PyTex version, the
canonical convention set, and (optionally) the phase. `to_dict()` yields the plain
data structure; `write_json()` writes the document to disk. Notice how much the
manifest makes explicit that a bare data file would leave implicit.

In [ ]:
manifest = ExperimentManifest.from_acquisition_geometry(
    acquisition, phase=phase, source_system="demo-ebsd-scope",
)
payload = manifest.to_dict()

print("schema id      :", payload["schema_id"])
print("schema version :", payload["schema_version"])
print("source system  :", payload["source_system"])
print("convention set :", payload["canonical_convention_set"])
print("phase recorded :", payload["phase"]["name"])
print("\ntop-level manifest keys:")
print("  " + ", ".join(payload.keys()))

## 3 · Round-trip and schema validation

The reproducibility contract has two clauses: the document must **survive a
round-trip** (write → read returns the same content), and it must **validate against
its published schema**. We check both. `validate_experiment_manifest` raises if the
payload violates `schemas/experiment_manifest.schema.json`; silence means success.

In [ ]:
tmp_dir = Path(tempfile.mkdtemp())
manifest_path = tmp_dir / "experiment.json"
manifest.write_json(manifest_path)

reloaded = read_experiment_manifest(manifest_path)
round_trips = reloaded.to_dict() == payload
print("write -> read reproduces the manifest exactly:", round_trips)
assert round_trips

validate_experiment_manifest(payload)   # raises on any schema violation
print("payload validates against the published schema: True")

# The quality metadata survives intact through JSON.
print("\nreloaded confidence:", reloaded.to_dict()["measurement_quality"]["confidence"])

### The manifest as a human-readable document

Because it is plain JSON, the manifest is greppable, diffable, and reviewable. A short
excerpt shows the calibration and quality blocks a downstream tool would read.

In [ ]:
excerpt = {
    "schema_id": payload["schema_id"],
    "modality": payload["modality"],
    "specimen_to_map": payload["specimen_to_map"],
    "calibration_record": payload["calibration_record"],
    "measurement_quality": payload["measurement_quality"],
}
print(json.dumps(excerpt, indent=2))

> **Good to know.**
>
> - CIF succeeded where dozens of file formats failed because it shipped a machine-readable
>   *dictionary* alongside the syntax: the IUCr adopted it in 1991, and a CIF names what its
>   numbers mean rather than assuming a reader who knows. That is the ancestry of the manifest in
>   this notebook.
> - The FAIR principles (2016) spell out the same thing for data generally — findable, accessible,
>   interoperable, reusable — and the interoperable clause is the expensive one: it requires that
>   frames, units and conventions travel *with* the numbers.
> - The failure this guards against is silent, not loud. A dataset with the tilt-axis convention
>   left implicit does not fail to load; it loads and gives an answer that is wrong by a
>   reflection.

## Summary and where to go next

- **`AcquisitionGeometry`** unifies frames, modality, calibration, and quality across
  EBSD/XRD/TEM instead of letting each subsystem reinvent them.
- **`ExperimentManifest`** freezes that into a **schema-versioned, JSON-serialisable**
  record that round-trips and validates — the interoperability floor for any
  cross-tool workflow.
- The same manifest family extends to import, benchmark, validation, and
  workflow-result records used throughout the pipeline notebooks.

**Next:** [Notebook 15](15_structure_diffraction_visualization_pipeline.ipynb) uses
workflow-result and validation manifests to stitch a full structure→diffraction
pipeline together, and [Notebook 07](07_ebsd_regular_grid_workflows.ipynb) consumes
acquisition geometry inside the EBSD grid workflows.